In [2]:
import os
import json
import hashlib
from urllib.parse import urlsplit
from tqdm import tqdm
import time
from torch.utils.data import DataLoader
from datasets import load_dataset


DATASET_NAME = "HuggingFaceFW/fineweb-2"
DATASET_SUBSET = "rus_Cyrl"
SPLIT = "train"

OUTPUT_DIR = "mini_fineweb2_rus_all_data"
RECORDS_PER_SHARD = 10_000
TEXTS_PER_SITE = 10

LOG_FILE = "progress.log"

def open_shard(output_dir: str, shard_idx: int):
    os.makedirs(output_dir, exist_ok=True)
    path = os.path.join(output_dir, f"part-{shard_idx:05d}.jsonl")
    return open(path, "a", encoding="utf-8"), path

/usr/local/lib/python3.10/dist-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
site_text_counts, total_processed, shard_idx, shard_size, total_iterated = {}, 0, 0, 0, 0

In [ ]:
dataset = load_dataset(
    DATASET_NAME,
    name=DATASET_SUBSET,
    split=SPLIT,
    streaming=True,
    columns=[
        "id",
        "dump",
        "file_path",
        "language_score",
        "minhash_cluster_size",
        "url",
        "text",
        "date",
    ],
)

loader = DataLoader(
    dataset,
    batch_size=None,
    num_workers=4,
    prefetch_factor=10,
)

shard_file, shard_path = open_shard(OUTPUT_DIR, shard_idx)

for doc in tqdm(loader, total=700_000_000):
    total_iterated += 1
    if total_iterated % 1000000 == 0:
        with open(LOG_FILE, "w") as f:
            f.write(str(total_iterated))
            
    text = doc.get("text") or ""
    if not text.strip():
        continue

    url = doc.get("url")
    site = ''
    try:
        parts = urlsplit(url)
        if not parts.scheme or not parts.netloc:
            continue
        site = f"{parts.scheme}://{parts.netloc}"
    except:
        continue

    current_site_count = site_text_counts.get(site, 0) if site else 0

    if current_site_count < TEXTS_PER_SITE:
        site_text_counts[site] = current_site_count + 1
    else:
        continue

    record = {
        "id": doc.get("id"),
        "url": url,
        "site": site,
        "sha256": hashlib.sha256(text.encode("utf-8")).hexdigest(), # sha256
        "date": doc.get("date"),
        "site_count": site_text_counts[site],
        "dump": doc.get("dump"),
        "file_path": doc.get("file_path"),
        "language_score": doc.get("language_score"),
        "minhash_cluster_size": doc.get("minhash_cluster_size"),
        "text": text,
    }

    shard_file.write(json.dumps(record, ensure_ascii=False) + "\n")
    shard_size += 1
    total_processed += 1

    if shard_size >= RECORDS_PER_SHARD:
        shard_file.close()
        shard_idx += 1
        shard_size = 0
        shard_file, shard_path = open_shard(OUTPUT_DIR, shard_idx)

shard_file.close()
        
print(f"Total processed: {total_processed}")
print(f"Sites with saved texts: {len(site_text_counts)}")

  0%|▎                                                                                                                                                                                             | 1067750/700000000 [03:10<36:57:46, 5252.49it/s]

In [5]:
print(total_iterated)

631499724


In [22]:
len(site_text_counts), total_processed, shard_idx

(7336186, 36678757, 3667)

In [ ]:
1